# M3 · Load dimensions

**Goal:** run `sql/01_dim_tenant.sql` through `sql/05_dim_date_hour.sql` in order. These are full-refresh idempotent upserts — re-running does not duplicate rows.

This is the data-loading half of what the old `bootstrap` mode did.

**Exit criterion:** row counts match (5 tenants, 638 vehicles, 633 devices, 294 drivers, ~3287 dates, 24 hour-bands).


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — dimension SQL files

In [ ]:
from accent_fleet.db.sql_loader import load_sql
DIM_FILES = [
    '01_dim_tenant.sql',
    '02_dim_vehicle.sql',
    '03_dim_device.sql',
    '04_dim_driver.sql',
    '05_dim_date_hour.sql',
]
for f in DIM_FILES:
    head = load_sql(f).splitlines()[:5]
    print(f"\n-- {f}")
    for l in head:
        print(l)


## 3. Execute — via the shared helper (same code path as production)

In [ ]:
import time
from accent_fleet.transforms import refresh_all_dimensions
t0 = time.time()
refresh_all_dimensions()
print(f"Dimensions refreshed in {time.time()-t0:.2f}s")


## 4. Inspect

In [ ]:
import pandas as pd
EXPECTED = {'dim_tenant': 5, 'dim_vehicle': 638, 'dim_device': 633, 'dim_driver': 294, 'dim_hour_band': 24}
rows = []
with get_engine().connect() as conn:
    for t in ['dim_tenant','dim_vehicle','dim_device','dim_driver','dim_date','dim_hour_band']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM warehouse.{t}')).scalar_one()
        rows.append({'table': t, 'rows': n})
df = pd.DataFrame(rows); display(df)
for t, expected in EXPECTED.items():
    got = int(df.loc[df['table']==t, 'rows'].iloc[0])
    assert got == expected or abs(got-expected) <= 5, f"{t}: expected ~{expected}, got {got}"
print("OK — dimensions populated.")


In [ ]:
# Sample rows for visual check
import pandas as pd
with get_engine().connect() as conn:
    for t in ['dim_tenant','dim_vehicle','dim_device','dim_driver']:
        print(f"\n=== warehouse.{t} (sample) ===")
        display(pd.read_sql(text(f'SELECT * FROM warehouse.{t} LIMIT 5'), conn))
